# 05 · Task 2 challenger — Agentic RAG (LangGraph) + ablations
**Owner:** B (Task-2 lead) · **Phase 2**

Prompts are **frozen at v2** (2026-11-23). They were tuned on the 48-query dev set with
`scripts/dev_agentic.py`, never on the eval set; the record is in `prompts/CHANGELOG.md`.
The intent tool and its threshold were chosen on dev data with `scripts/choose_intent_tool.py`.
This notebook runs the frozen system on the full eval set, then the ablations below.

| Run tag | What changes | Question it answers |
|---|---|---|
| `agentic_full` | everything on | headline challenger |
| `agentic_no_validation` | `use_validation=False` | does validate→revise→escalate help? |
| `agentic_no_grading` | `use_grading=False` | does grading + query rewrite help? |
| `agentic_no_hybrid` | `use_hybrid=False` | does BM25 fusion help (typos, real wording)? |
| `agentic_no_classifier` | `use_classifier_tool=False` | does the Task-1 tool help vs LLM-predicted intent? |

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: clone the team repo once per session, install requirements, load API keys from Secrets
    REPO_URL = "https://github.com/sh1or1tama1/banking-support-agent.git"
    if not os.path.exists("/content/banking-support-agent"):
        subprocess.run(["git", "clone", REPO_URL, "/content/banking-support-agent"], check=True)
    os.chdir("/content/banking-support-agent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
# os.environ["LLM_PROVIDER"] = "openai"; os.environ["LLM_MODEL"] = "<same model as notebook 04>"
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import joblib, json, pandas as pd
from IPython.display import display
from src import config as C
from src.rag.retriever import build_kb, DenseRetriever, BM25Retriever
from src.rag.agentic_rag import AgenticRAG, AgenticConfig
from src.classify.tfidf_lr import as_tool
from src.llm.client import LLMClient
from src.eval.harness import run_system, summarize, escalation_confusion, load_run

LIMIT = int(os.getenv("LIMIT")) if os.getenv("LIMIT") else None   # None = full eval set
PROMPTS = "v2"          # frozen, see prompts/CHANGELOG.md
TOOL, THRESHOLD = "bert", 0.99   # chosen on dev data: outputs/tables/intent_tool_threshold_sweep.csv

splits = pd.read_parquet(C.SPLITS_FILE)
ev = pd.read_parquet(C.EVAL_SETS_FILE)
kb = build_kb(splits)
dense, bm25 = DenseRetriever(kb), BM25Retriever(kb)
# Intent tool = Task-1 classifier called by the agent (see TOOL above)
if TOOL == "bert":
    from src.classify.bert import BertIntent
    classifier = BertIntent().as_tool()
else:
    classifier = as_tool(joblib.load(C.OUTPUTS / "tfidf_lr.joblib"))
base = dict(intent_conf_threshold=THRESHOLD)
llm = LLMClient()

/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1123 02:18:34.543000 71463 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9409.80it/s]

Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Batches:   1%|          | 1/132 [00:00<01:24,  1.55it/s]

... (63 outputs left out)

Batches:  98%|█████████▊| 130/132 [00:09<00:00, 18.76it/s]

Batches: 100%|██████████| 132/132 [00:09<00:00, 13.80it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7347.59it/s]

## 1. One query, full trace (use this to debug and to make the demo slide)

In [2]:
agent = AgenticRAG(llm, dense, bm25, classifier=classifier, cfg=AgenticConfig(**base), prompt_version=PROMPTS)
out = agent.answer("i lost my card yesterday and i also want to know how to send money to my brother")
print(out["action"], "|", out["answer"][:300])
for step in out["trace"]:
    print("-", step["step"], "→", str(step["output"])[:150])
print({k: out[k] for k in ("n_llm_calls", "input_tokens", "output_tokens", "cost_usd", "wall_time_s")})

escalate | I'm sorry, I can't help with that here. I'm connecting you with one of our human agents, who can assist you further.
- analyze → {
  "intent": "block_card",
  "is_banking": true,
  "needs_clarification": false,
  "sub_questions": [
    "How do I send money to my brother?"
  ],
 
- tool:intent_classifier → ['cancel_card', 0.634]
- retrieve(round 1, filter=False) → ['bx_04933', 'bx_04376', 'bx_04256']
- grade → {
  "relevant": [1, 2, 3],
  "reason": "The snippets explain what to do when a customer has lost their card and needs to block it, matching part of th
- generate → {"action":"answer","answer":"I’m sorry about your lost card. Here’s what to do:\n1) Block the card ASAP: contact {{Customer Support Phone Number}} (24
- validate → {
  "invented_fact": "",
  "claims_action": false,
  "asks_secret_in_chat": false,
  "missed_part": "",
  "wrong_action": true,
  "verdict": "fail",
 
{'n_llm_calls': 4, 'input_tokens': 3800, 'output_tokens': 372, 'cost_usd': 0.001226, 'wall_time_s

## 2. Full system + ablations

In [3]:
variants = {
    "agentic_full": AgenticConfig(**base),
    "agentic_no_validation": AgenticConfig(**base, use_validation=False),
    "agentic_no_grading": AgenticConfig(**base, use_grading=False),
    "agentic_no_hybrid": AgenticConfig(**base, use_hybrid=False),
    "agentic_no_classifier": AgenticConfig(**base, use_classifier_tool=False),
}
runs = {}
for tag, cfg in variants.items():
    runs[tag] = run_system(AgenticRAG(llm, dense, bm25, classifier=classifier, cfg=cfg, name=tag, prompt_version=PROMPTS),
                           ev, tag=tag, limit=LIMIT)

agentic_full:   0%|          | 0/436 [00:00<?, ?it/s]

agentic_full:   0%|          | 1/436 [00:04<32:48,  4.53s/it]

agentic_full:   0%|          | 2/436 [00:09<35:09,  4.86s/it]

agentic_full:   1%|          | 3/436 [00:14<36:20,  5.03s/it]

agentic_full:   1%|          | 4/436 [00:19<35:43,  4.96s/it]

agentic_full:   1%|          | 5/436 [00:24<35:07,  4.89s/it]

... (1,761 outputs left out)

agentic_no_classifier:  99%|█████████▉| 433/436 [13:21<00:04,  1.45s/it]

agentic_no_classifier: 100%|█████████▉| 434/436 [13:24<00:03,  1.79s/it]

agentic_no_classifier: 100%|█████████▉| 435/436 [13:26<00:02,  2.07s/it]

agentic_no_classifier: 100%|██████████| 436/436 [13:26<00:00,  1.85s/it]

In [4]:
rows = []
for tag, run in runs.items():
    s = summarize(run, ev).reset_index()
    s.insert(0, "system", tag)
    rows.append(s)
abl = pd.concat(rows)
display(abl.pivot_table(index="eval_set", columns="system", values="behaviour_accuracy").round(3))
display(abl.groupby("system")[["mean_llm_calls", "mean_tokens", "total_cost_usd", "median_wall_s"]].mean().round(3))
abl.to_csv(C.TABLES / "task2_agentic_ablations.csv", index=False)

system,agentic_full,agentic_no_classifier,agentic_no_grading,agentic_no_hybrid,agentic_no_validation
eval_set,,,,,
ambiguous,0.833,0.800,0.767,0.833,0.833
b77_real_inscope,0.600,0.640,0.610,0.570,0.610
b77_unsupported,0.525,0.516,0.525,0.533,0.525
bitext_core,0.942,0.952,0.942,0.952,0.942
bitext_hard,0.920,0.900,0.920,0.920,0.920
out_of_domain,0.900,0.933,0.967,0.933,0.900


,mean_llm_calls,mean_tokens,total_cost_usd,median_wall_s
system,,,,
agentic_full,3.410,3012.381,0.066,3.499
agentic_no_classifier,3.391,2989.967,0.066,1.619
agentic_no_grading,2.372,2364.615,0.054,0.466
agentic_no_hybrid,3.386,2983.710,0.066,2.400
agentic_no_validation,2.899,2358.644,0.053,0.070


## 3. Failure analysis — read 10 traces where the agent got the behaviour wrong

In [5]:
full = runs["agentic_full"].merge(ev[["eval_id", "expected_action"]], on="eval_id")
bad = full[full["action"] != full["expected_action"]]
for _, r in bad.head(10).iterrows():
    print(f"\n[{r['eval_set']}] expected={r['expected_action']} got={r['action']}\nQ: {r['query']}\nA: {str(r['answer'])[:200]}")
    for st in r["trace"]:
        print("   -", st["step"], "→", str(st["output"])[:120])


[bitext_core] expected=answer got=clarify
Q: I'd like to see the fucking card nanual fee
A: I can help with that. Which card are you referring to (card name/type and issuer, if you know it)?
   - analyze → {
  "intent": "check_card_annual_fee",
  "is_banking": true,
  "needs_clarification": false,
  "sub_questions": [],
  "s
   - tool:intent_classifier → ['check_card_annual_fee', 0.996]
   - retrieve(round 1, filter=True) → ['bx_09826', 'bx_09706', 'bx_09416']
   - grade → {
  "relevant": [1, 2, 3],
  "reason": "All snippets address how to view or check the card annual fee."
}
   - generate → {"action":"clarify","answer":"I can help with that. Which card are you referring to (card name/type and issuer, if you k

[bitext_core] expected=answer got=clarify
Q: I would like to cancela profile, will you help me?
A: Sure—I can help. When you say “profile,” do you mean your **online account/profile** (website/app) or a **bank account** you want to close?
   - analyze → {
  "intent": "close_ac